# MKUU Foundation Model — Google Colab T4 Pilot

Run the cells from top to bottom. This workflow restores and verifies the approved Swahili source snapshot, cleans and splits it, trains the SentencePiece tokenizer, performs FP16 pretraining, saves a rolling resumable checkpoint to Google Drive, and evaluates held-out validation/test samples.

**Requirements:** a Colab GPU runtime (T4 verified in the project preflight) and Google Drive space for a large checkpoint. This pilot does not guarantee full-scale convergence. It does not modify the GitHub repository during training.

In [ ]:
from pathlib import Path
import subprocess

REPO = Path('/content/-mkuu-foundation-model-')
BRANCH = 'colab-t4-pilot'
if not REPO.exists():
    subprocess.run(['git', 'clone', '-b', BRANCH, 'https://github.com/maxmkuu-eng/-mkuu-foundation-model-.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)
print('Repository ready:', REPO)
print('Branch:', subprocess.check_output(['git','-C',str(REPO),'branch','--show-current'], text=True).strip())

In [ ]:
%cd /content/-mkuu-foundation-model-
!pip -q install 'sentencepiece>=0.2.0' 'PyYAML>=6.0' 'numpy>=1.24' 'requests>=2.31'
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU haipatikani. Chagua Runtime > Change runtime type > T4 GPU kisha reconnect.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
!python -m py_compile scripts/colab_t4_pilot.py

## Run the complete pilot

This mounts Drive and asks Google to authorize access. Check that your Drive has enough free space before continuing. The script keeps only one rolling full checkpoint (`last.pt`) to limit storage growth. If Colab disconnects, reconnect, run the setup cells, and run this cell again; it resumes from Drive automatically.

In [ ]:
!python scripts/colab_t4_pilot.py --mount-drive --steps 1000 --save-every 100 --sequence-length 512 --eval-sequences 256

## Results

The final report is saved to `MyDrive/MKUU-Foundation-Model/pilot_report.json`; the resumable checkpoint is in `MyDrive/MKUU-Foundation-Model/checkpoints/last.pt`. Evaluation is limited to the first 256 sequences from each held-out split for this pilot. Do not treat this pilot as a production-ready language model.